# 02. Numerical Feature Engineering
### Notebook 2 – Numerical Feature Engineering

**Dataset**: Online Retail Transactions (`data.csv`)

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('data.csv', encoding='ISO-8859-1')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


# 1. Mathematical Transformations

Mathematical transformations combine two or more numerical columns using basic arithmetic to create a new feature that carries more meaning than either column alone.

**When to use:** whenever a raw column by itself does not capture the real quantity a model needs, but combining it with another column does.

**Why:** models cannot learn arithmetic relationships between columns on their own unless the relationship is explicitly given as a feature.

# 2. Addition

Adding two or more numeric columns to represent a combined total.

**When to use:** when several columns represent parts of a whole, such as combining multiple cost components into one total cost.

**Why:** a single combined feature is often more predictive than its individual parts, and reduces the number of columns the model has to relate to each other.

In [2]:
df['Quantity_plus_Price'] = df['Quantity'] + df['UnitPrice']
df[['Quantity', 'UnitPrice', 'Quantity_plus_Price']].head()

,Quantity,UnitPrice,Quantity_plus_Price
0,6,2.55,8.55
1,6,3.39,9.39
2,8,2.75,10.75
3,6,3.39,9.39
4,6,3.39,9.39


# 3. Subtraction

Subtracting one numeric column from another to represent a change or gap.

**When to use:** when the difference between two values matters more than their absolute levels, such as change over time or gap between expected and actual values.

**Why:** models often perform better with the difference already computed, since it removes the need for the model to infer the relationship itself.

In [3]:
customer_avg_price = df.groupby('CustomerID')['UnitPrice'].transform('mean')
df['Price_diff_from_avg'] = df['UnitPrice'] - customer_avg_price
df[['CustomerID', 'UnitPrice', 'Price_diff_from_avg']].head()

,CustomerID,UnitPrice,Price_diff_from_avg
0,17850.0,2.55,-1.374712
1,17850.0,3.39,-0.534712
2,17850.0,2.75,-1.174712
3,17850.0,3.39,-0.534712
4,17850.0,3.39,-0.534712


# 4. Multiplication

Multiplying two numeric columns together to represent a combined effect.

**When to use:** when the true business quantity is the product of two columns, such as price and quantity giving revenue.

**Why:** the interaction between two variables can be more predictive than either variable alone, and many models cannot learn multiplicative interactions without being told.

In [4]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df[['Quantity', 'UnitPrice', 'TotalPrice']].head()

,Quantity,UnitPrice,TotalPrice
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


# 5. Division

Dividing one numeric column by another to normalize one quantity relative to another.

**When to use:** when comparing two values only makes sense relative to a base, such as price per unit or spend per transaction.

**Why:** division removes scale differences between rows, making values comparable across customers, products, or time periods.

In [5]:
customer_totals = df.groupby('CustomerID').agg(TotalSpend=('TotalPrice', 'sum'), TotalOrders=('InvoiceNo', 'nunique'))
customer_totals['AvgSpendPerOrder'] = customer_totals['TotalSpend'] / customer_totals['TotalOrders']
customer_totals.head()

,TotalSpend,TotalOrders,AvgSpendPerOrder
CustomerID,,,
12346.0,0.00,2,0.000000
12347.0,4310.00,7,615.714286
12348.0,1797.24,4,449.310000
12349.0,1757.55,1,1757.550000
12350.0,334.40,1,334.400000


# 6. Ratios

A ratio expresses the relationship between two related quantities, usually as a fraction.

**When to use:** when the relative size of one quantity to another carries more signal than either value alone, such as return rate or discount ratio.

**Why:** ratios are scale-independent, so they generalize better across customers or products with very different absolute values.

In [6]:
returns = df[df['Quantity'] < 0].groupby('CustomerID')['InvoiceNo'].nunique().rename('ReturnOrders')
orders = df.groupby('CustomerID')['InvoiceNo'].nunique().rename('TotalOrders')
customer_ratio = pd.concat([orders, returns], axis=1).fillna(0)
customer_ratio['ReturnRate'] = customer_ratio['ReturnOrders'] / customer_ratio['TotalOrders']
customer_ratio.head()

,TotalOrders,ReturnOrders,ReturnRate
CustomerID,,,
12346.0,2,1.0,0.5
12347.0,7,0.0,0.0
12348.0,4,0.0,0.0
12349.0,1,0.0,0.0
12350.0,1,0.0,0.0


# 7. Percentages

A percentage expresses a ratio scaled to a base of 100, making it easier to interpret.

**When to use:** when stakeholders or the model benefit from a standardized, human readable scale, such as percentage of total revenue contributed by a product.

**Why:** percentages are intuitive and comparable across groups of very different sizes.

In [7]:
product_revenue = df.groupby('StockCode')['TotalPrice'].sum()
total_revenue = product_revenue.sum()
product_pct = (product_revenue / total_revenue) * 100
product_pct.sort_values(ascending=False).head()

StockCode
DOT       2.115827
22423     1.690259
47566     1.008469
85123A    1.004278
85099B    0.947460
Name: TotalPrice, dtype: float64

# 8. Differences

A difference feature captures the change between two related values, often across time or between an entity and a reference point.

**When to use:** when trend or change matters more than the raw value, such as change in spend between two periods.

**Why:** absolute values often hide the direction and magnitude of change, which is frequently the actual signal a model needs.

In [8]:
monthly_spend = df.groupby([df['CustomerID'], df['InvoiceDate'].dt.to_period('M')])['TotalPrice'].sum().reset_index()
monthly_spend.columns = ['CustomerID', 'Month', 'MonthlySpend']
monthly_spend['SpendDiff'] = monthly_spend.groupby('CustomerID')['MonthlySpend'].diff()
monthly_spend.head()

,CustomerID,Month,MonthlySpend,SpendDiff
0,12346.0,2011-01,0.00,NaN
1,12347.0,2010-12,711.79,NaN
2,12347.0,2011-01,475.39,-236.40
3,12347.0,2011-04,636.25,160.86
4,12347.0,2011-06,382.52,-253.73


# 9. Aggregations

Aggregations summarize multiple rows belonging to the same group into a single statistic, such as sum, mean, count, min or max.

**When to use:** when the raw data is at a finer granularity than the entity the model needs to reason about, such as transaction-level data being aggregated to the customer level.

**Why:** aggregation turns repeated, granular records into meaningful entity-level features, and is one of the most common sources of predictive power in tabular data.

In [17]:
customer_features = df.groupby('CustomerID').agg(
    TotalSpend=('TotalPrice', 'sum'),
    AvgUnitPrice=('UnitPrice', 'mean'),
    TotalQuantity=('Quantity', 'sum'),
    NumOrders=('InvoiceNo', 'nunique'),
    NumProducts=('StockCode', 'nunique')
)
customer_features.head()

,TotalSpend,AvgUnitPrice,TotalQuantity,NumOrders,NumProducts
CustomerID,,,,,
12346.0,0.00,1.040000,0,2,1
12347.0,4310.00,2.644011,2458,7,103
12348.0,1797.24,5.764839,2341,4,22
12349.0,1757.55,8.289041,631,1,73
12350.0,334.40,3.841176,197,1,17


# 10. Log Transformation

Log transformation compresses large values and expands small ones, reducing the effect of extreme outliers and right skew.

**When to use:** when a numeric feature is heavily right skewed, such as monetary values or counts with a long tail of large outliers.

**Why:** many models assume features are roughly symmetric or normally distributed, and log transformation stabilizes variance and improves model performance on skewed data.

In [10]:
df['TotalPrice_log'] = np.log1p(df['TotalPrice'].clip(lower=0))
df[['TotalPrice', 'TotalPrice_log']].describe()

,TotalPrice,TotalPrice_log
count,541909.000000,541909.000000
mean,17.987795,2.280430
std,378.810824,1.063879
min,-168469.600000,0.000000
25%,3.400000,1.481605
50%,9.750000,2.374906
75%,17.400000,2.912351
max,168469.600000,12.034517


# 11. Square Root Transformation

Square root transformation reduces skew similarly to log transformation but with a gentler effect, and can handle zero values directly.

**When to use:** when a feature is moderately right skewed and contains zeros, where a log transform would need an offset.

**Why:** it is a milder correction than log, useful when full log compression would over correct the distribution.

In [11]:
df['Quantity_sqrt'] = np.sqrt(df['Quantity'].clip(lower=0))
df[['Quantity', 'Quantity_sqrt']].describe()

,Quantity,Quantity_sqrt
count,541909.000000,541909.000000
mean,9.552250,2.416361
std,218.081158,2.146526
min,-80995.000000,0.000000
25%,1.000000,1.000000
50%,3.000000,1.732051
75%,10.000000,3.162278
max,80995.000000,284.596205


# 12. Power Transformation

Power transformation raises a feature to a power, such as squaring it, to either amplify large values or, with fractional powers, further correct skew beyond what log or square root achieve.

**When to use:** when the relationship between a feature and the target is non-linear, or when log and square root transforms are insufficient to normalize the distribution.

**Why:** it gives explicit control over how strongly large or small values are emphasized, and can reveal non-linear relationships to linear models.

In [12]:
from sklearn.preprocessing import PowerTransformer
pt = PowerTransformer(method='yeo-johnson')
df['UnitPrice_power'] = pt.fit_transform(df[['UnitPrice']])
df[['UnitPrice', 'UnitPrice_power']].describe()

,UnitPrice,UnitPrice_power
count,541909.000000,5.419090e+05
mean,4.611114,4.195791e-18
std,96.759853,1.000001e+00
min,-11062.060000,-3.380760e+02
25%,1.250000,-3.943001e-02
50%,2.080000,-2.735903e-02
75%,4.130000,1.463068e-03
max,38970.000000,2.867979e+02


# 13. Absolute Difference

The absolute difference measures the magnitude of change between two values, ignoring direction.

**When to use:** when only the size of the deviation matters, not whether it increased or decreased, such as measuring how far a price is from a benchmark.

**Why:** it prevents positive and negative deviations from cancelling out when aggregated, and is useful for detecting outliers or anomalies.

In [13]:
category_avg_price = df.groupby('StockCode')['UnitPrice'].transform('mean')
df['AbsPriceDiff'] = (df['UnitPrice'] - category_avg_price).abs()
df[['StockCode', 'UnitPrice', 'AbsPriceDiff']].head()

,StockCode,UnitPrice,AbsPriceDiff
0,85123A,2.55,0.553238
1,71053,3.39,1.387493
2,84406B,2.75,1.504966
3,84029G,3.39,1.786540
4,84029E,3.39,1.678208


# 14. Relative Difference

Relative difference expresses a difference as a proportion of a reference value rather than an absolute amount.

**When to use:** when the same absolute difference means different things depending on scale, such as a price change of 1 unit meaning more for a cheap product than an expensive one.

**Why:** it allows fair comparison of change across entities with very different baseline magnitudes.

In [18]:
df['RelPriceDiff'] = (df['UnitPrice'] - category_avg_price) / category_avg_price
df[['StockCode', 'UnitPrice', 'RelPriceDiff']].head()

,StockCode,UnitPrice,RelPriceDiff
0,85123A,2.55,-0.178278
1,71053,3.39,-0.290423
2,84406B,2.75,-0.353696
3,84029G,3.39,-0.345122
4,84029E,3.39,-0.331125


# 15. Rate Features

A rate feature expresses a quantity per unit of time or per unit of another dimension, such as orders per month or spend per day.

**When to use:** when comparing entities that have been observed over different lengths of time or different amounts of activity.

**Why:** raw totals are biased toward entities with longer histories or more activity, while rates normalize for that and allow fair comparison.

In [15]:
customer_span = df.groupby('CustomerID')['InvoiceDate'].agg(['min', 'max'])
customer_span['DaysActive'] = (customer_span['max'] - customer_span['min']).dt.days.replace(0, 1)
customer_orders = df.groupby('CustomerID')['InvoiceNo'].nunique()
order_rate = (customer_orders / customer_span['DaysActive']).rename('OrdersPerDay')
order_rate.head()

CustomerID
12346.0    2.000000
12347.0    0.019178
12348.0    0.014184
12349.0    1.000000
12350.0    1.000000
Name: OrdersPerDay, dtype: float64

# 16. Normalized Features

Normalization rescales a feature into a fixed range, typically 0 to 1, based on its minimum and maximum values.

**When to use:** when features are on very different scales and the model or algorithm is sensitive to magnitude, such as distance based models or gradient based optimization.

**Why:** it prevents features with naturally larger numeric ranges from dominating the model simply because of their scale, rather than their actual importance.

In [16]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[['Quantity_norm', 'UnitPrice_norm']] = scaler.fit_transform(df[['Quantity', 'UnitPrice']])
df[['Quantity', 'Quantity_norm', 'UnitPrice', 'UnitPrice_norm']].head()

,Quantity,Quantity_norm,UnitPrice,UnitPrice_norm
0,6,0.500037,2.55,0.221150
1,6,0.500037,3.39,0.221167
2,8,0.500049,2.75,0.221154
3,6,0.500037,3.39,0.221167
4,6,0.500037,3.39,0.221167
